# Goal-Conditioned Planning with World Models

World models enable **planning without reward functions**. Given a current
observation and a goal observation, the model imagines the consequences of
different action sequences and picks the one that gets closest to the goal.

This notebook covers:
1. How CEM (Cross-Entropy Method) planning works
2. Basic goal-conditioned planning
3. Hierarchical planning for long-horizon tasks
4. Analyzing and visualizing plans
5. Planning with different action spaces

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/07_plan_robot_actions.ipynb)

**Time to run:** ~8 minutes on Colab T4

In [ ]:
!pip install "worldkit[train,envs]" -q

## 1. How CEM Planning Works

The **Cross-Entropy Method** (CEM) is WorldKit's default planner:

1. **Sample** `n_candidates` random action sequences
2. **Simulate** each sequence through the world model (in latent space)
3. **Score** each by distance to the goal in latent space
4. **Select** the top `n_elite` sequences
5. **Refine**: fit a Gaussian to the elite set, resample, and repeat

After `n_iterations`, return the best action sequence found.

The key insight: all simulation happens in latent space, so it's fast.
No pixel rendering needed.

## 2. Setup: Train a Model

We'll create a simple environment where planning is visually intuitive:
an agent (dot) that can move in 2D toward a target position.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch


def render_agent(x, y, size=96):
    """Render an agent (blue dot) and grid."""
    frame = np.full((size, size, 3), 240, dtype=np.uint8)  # light background
    # Grid lines
    for i in range(0, size, size // 6):
        frame[i, :] = [200, 200, 200]
        frame[:, i] = [200, 200, 200]
    # Agent (blue circle)
    yy, xx = np.ogrid[:size, :size]
    mask = (xx - int(x)) ** 2 + (yy - int(y)) ** 2 <= 36
    frame[mask] = [50, 100, 200]
    return frame


def generate_navigation_data(n_episodes=300, timesteps=16, size=96):
    """Agent moves in response to 2D actions."""
    rng = np.random.RandomState(42)
    all_pix = []
    all_act = []

    for _ in range(n_episodes):
        x = rng.uniform(10, size - 10)
        y = rng.uniform(10, size - 10)
        frames = []
        actions_ep = []

        for t in range(timesteps):
            frames.append(render_agent(x, y, size))
            # Random action: dx, dy
            ax = rng.uniform(-4, 4)
            ay = rng.uniform(-4, 4)
            actions_ep.append([ax, ay])
            x = np.clip(x + ax, 6, size - 6)
            y = np.clip(y + ay, 6, size - 6)

        all_pix.append(np.array(frames))
        all_act.append(np.array(actions_ep, dtype=np.float32))

    return np.array(all_pix, dtype=np.uint8), np.array(all_act)


# Generate and save
pixels, actions = generate_navigation_data()

with h5py.File("nav_data.h5", "w") as f:
    f.create_dataset("pixels", data=pixels, compression="gzip")
    f.create_dataset("actions", data=actions, compression="gzip")

print(f"Data: {pixels.shape[0]} episodes, actions: {actions.shape}")

# Visualize one episode
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(pixels[0, i * 2])
    ax.set_title(f"t={i * 2}")
    ax.axis("off")
plt.suptitle("Sample episode: agent moving in 2D", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from worldkit import WorldModel

model = WorldModel.train(
    data="nav_data.h5",
    config="nano",
    epochs=25,
    batch_size=32,
    action_dim=2,
    device="auto",
)

print(f"\nTrained: {model.num_params:,} params")

## 3. Basic Planning

Given a start position and a goal position, the planner finds an
action sequence to move the agent from start to goal.

In [ ]:
# Create start and goal frames
start_frame = render_agent(20, 20)  # Top-left
goal_frame = render_agent(76, 76)   # Bottom-right

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(start_frame); axes[0].set_title("Start (20, 20)"); axes[0].axis("off")
axes[1].imshow(goal_frame); axes[1].set_title("Goal (76, 76)"); axes[1].axis("off")
plt.tight_layout()
plt.show()

# Plan!
plan = model.plan(
    current_state=start_frame,
    goal_state=goal_frame,
    max_steps=15,
    n_candidates=300,
    n_elite=30,
    n_iterations=5,
    action_space={"low": -4.0, "high": 4.0},
)

print(f"Planned {len(plan.actions)} actions")
print(f"Expected cost:       {plan.expected_cost:.4f}")
print(f"Success probability: {plan.success_probability:.4f}")
print(f"Planning time:       {plan.planning_time_ms:.1f} ms")

In [ ]:
# Visualize the planned actions
actions_arr = np.array(plan.actions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Action components over time
steps = range(len(actions_arr))
axes[0].plot(steps, actions_arr[:, 0], "o-", color="steelblue", label="dx")
axes[0].plot(steps, actions_arr[:, 1], "s-", color="coral", label="dy")
axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Action value")
axes[0].set_title("Planned actions over time")
axes[0].legend()

# Simulate the trajectory
x, y = 20.0, 20.0
traj_x, traj_y = [x], [y]
for a in plan.actions:
    a = np.array(a).flatten()
    x = np.clip(x + a[0], 6, 90)
    y = np.clip(y + a[1], 6, 90)
    traj_x.append(x)
    traj_y.append(y)

axes[1].plot(traj_x, traj_y, "o-", color="steelblue", markersize=6, label="Planned path")
axes[1].plot(20, 20, "*", markersize=15, color="green", label="Start")
axes[1].plot(76, 76, "*", markersize=15, color="red", label="Goal")
axes[1].set_xlim(0, 96)
axes[1].set_ylim(0, 96)
axes[1].set_aspect("equal")
axes[1].set_title("Planned trajectory")
axes[1].legend()
axes[1].invert_yaxis()  # Match image coordinates

plt.tight_layout()
plt.show()

## 4. Planning with Different Goals

The same model can plan to any goal — no retraining needed.
Let's plan from the center to each corner.

In [ ]:
center = render_agent(48, 48)
goals = {
    "Top-left (15, 15)": render_agent(15, 15),
    "Top-right (80, 15)": render_agent(80, 15),
    "Bottom-left (15, 80)": render_agent(15, 80),
    "Bottom-right (80, 80)": render_agent(80, 80),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ["steelblue", "coral", "mediumseagreen", "goldenrod"]

for ax, (goal_name, goal_frame), color in zip(axes, goals.items(), colors):
    plan = model.plan(
        current_state=center,
        goal_state=goal_frame,
        max_steps=12,
        n_candidates=300,
        n_elite=30,
        n_iterations=5,
        action_space={"low": -4.0, "high": 4.0},
    )

    # Simulate trajectory
    x, y = 48.0, 48.0
    traj_x, traj_y = [x], [y]
    for a in plan.actions:
        a = np.array(a).flatten()
        x = np.clip(x + a[0], 6, 90)
        y = np.clip(y + a[1], 6, 90)
        traj_x.append(x)
        traj_y.append(y)

    ax.imshow(np.full((96, 96, 3), 240, dtype=np.uint8), alpha=0.3)
    ax.plot(traj_x, traj_y, "o-", color=color, markersize=4)
    ax.plot(48, 48, "*", markersize=15, color="green")
    goal_xy = goal_name.split("(")[1].rstrip(")")
    gx, gy = [int(v) for v in goal_xy.split(", ")]
    ax.plot(gx, gy, "*", markersize=15, color="red")
    ax.set_title(f"Goal: {goal_name}\ncost={plan.expected_cost:.3f}")
    ax.set_xlim(0, 96)
    ax.set_ylim(0, 96)
    ax.invert_yaxis()
    ax.set_aspect("equal")

plt.suptitle("Planning from center to each corner", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Hierarchical Planning

For long-horizon tasks, **hierarchical planning** breaks the problem into
subgoals. The high-level planner sets intermediate waypoints, and
the low-level planner connects them.

This is useful when the goal is far away or requires navigating around obstacles.

In [ ]:
# Plan a long-distance move: corner to corner
start = render_agent(10, 10)
goal = render_agent(86, 86)

# Hierarchical plan
h_plan = model.hierarchical_plan(
    current_state=start,
    goal_state=goal,
    max_subgoals=3,
    steps_per_subgoal=10,
    n_candidates=300,
    n_elite=30,
    n_iterations=5,
    action_space={"low": -4.0, "high": 4.0},
)

print(f"Total actions: {len(h_plan.actions)}")
print(f"Subgoals: {len(h_plan.subgoals)}")
print(f"Planning time: {h_plan.total_planning_time_ms:.1f} ms")

for i, seg in enumerate(h_plan.segment_plans):
    print(f"  Segment {i}: {len(seg.actions)} actions, cost={seg.expected_cost:.4f}")

In [ ]:
# Compare flat vs hierarchical planning
flat_plan = model.plan(
    current_state=start,
    goal_state=goal,
    max_steps=30,
    n_candidates=300,
    n_elite=30,
    n_iterations=5,
    action_space={"low": -4.0, "high": 4.0},
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, plan_data, title in [
    (axes[0], flat_plan.actions, f"Flat planning ({len(flat_plan.actions)} steps)"),
    (axes[1], h_plan.actions, f"Hierarchical ({len(h_plan.actions)} steps, {len(h_plan.subgoals)} subgoals)"),
]:
    x, y = 10.0, 10.0
    traj_x, traj_y = [x], [y]
    for a in plan_data:
        a = np.array(a).flatten()
        x = np.clip(x + a[0], 6, 90)
        y = np.clip(y + a[1], 6, 90)
        traj_x.append(x)
        traj_y.append(y)

    ax.plot(traj_x, traj_y, "o-", color="steelblue", markersize=3, alpha=0.7)
    ax.plot(10, 10, "*", markersize=15, color="green", label="Start")
    ax.plot(86, 86, "*", markersize=15, color="red", label="Goal")
    ax.set_xlim(0, 96)
    ax.set_ylim(0, 96)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend()

plt.suptitle("Flat vs hierarchical planning", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nFlat cost:         {flat_plan.expected_cost:.4f}  ({flat_plan.planning_time_ms:.0f} ms)")
print(f"Hierarchical cost: {sum(s.expected_cost for s in h_plan.segment_plans):.4f}  ({h_plan.total_planning_time_ms:.0f} ms)")

## 6. Tuning the Planner

The CEM planner has several knobs. Let's see how they affect planning quality.

In [ ]:
import time

start = render_agent(15, 48)
goal = render_agent(80, 48)

# Vary n_candidates
candidate_counts = [50, 100, 200, 500, 1000]
costs = []
times = []

for n in candidate_counts:
    t0 = time.time()
    p = model.plan(
        start, goal,
        max_steps=10,
        n_candidates=n,
        n_elite=max(10, n // 10),
        n_iterations=5,
        action_space={"low": -4.0, "high": 4.0},
    )
    elapsed = time.time() - t0
    costs.append(p.expected_cost)
    times.append(elapsed * 1000)
    print(f"n_candidates={n:>4}: cost={p.expected_cost:.4f}, time={elapsed*1000:.0f}ms")

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(candidate_counts, costs, "o-", color="steelblue", label="Expected cost")
ax2.plot(candidate_counts, times, "s-", color="coral", label="Planning time")

ax1.set_xlabel("Number of candidates")
ax1.set_ylabel("Expected cost", color="steelblue")
ax2.set_ylabel("Planning time (ms)", color="coral")
ax1.set_title("Quality vs speed tradeoff")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

## 7. Planning in Latent Space

Let's visualize what the planner actually does — compute latent
trajectories and pick the one closest to the goal.

In [ ]:
# Encode start, goal, and a few intermediate points
positions = [(15, 15), (48, 15), (80, 15), (80, 48), (80, 80), (48, 80), (15, 80), (48, 48)]
frames = [render_agent(x, y) for x, y in positions]
latents = np.array([model.encode(f) for f in frames])

# Reduce to 2D for visualization
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
z2d = pca.fit_transform(latents)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: frames
for i, ((x, y), frame) in enumerate(zip(positions, frames)):
    axes[0].imshow(frame, extent=[x-12, x+12, y+12, y-12])
axes[0].set_xlim(0, 96)
axes[0].set_ylim(96, 0)
axes[0].set_title("Pixel space")
axes[0].set_aspect("equal")

# Right: latent space
axes[1].scatter(z2d[:, 0], z2d[:, 1], s=100, c=range(len(z2d)), cmap="viridis", zorder=5)
for i, (x, y) in enumerate(positions):
    axes[1].annotate(f"({x},{y})", (z2d[i, 0], z2d[i, 1]),
                     fontsize=8, ha="center", va="bottom")
axes[1].set_title("Latent space (PCA)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.suptitle("Spatial layout preserved in latent space", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show the planned latent trajectory alongside the actual trajectory
start = render_agent(15, 48)
goal = render_agent(80, 48)

plan = model.plan(
    start, goal,
    max_steps=10,
    n_candidates=300,
    n_elite=30,
    n_iterations=5,
    action_space={"low": -4.0, "high": 4.0},
)

# Predict the latent trajectory
pred = model.predict(start, actions=plan.actions, return_latents=True)

# Project to 2D using the same PCA
z_start = model.encode(start)
z_goal = model.encode(goal)
z_traj = pred.latent_trajectory

# Combine for PCA
all_z = np.vstack([z_start.reshape(1, -1), z_traj, z_goal.reshape(1, -1)])
pca2 = PCA(n_components=2)
all_2d = pca2.fit_transform(all_z)

fig, ax = plt.subplots(figsize=(8, 6))
# Trajectory
ax.plot(all_2d[1:-1, 0], all_2d[1:-1, 1], "o-", color="steelblue",
        markersize=6, label="Planned trajectory")
# Start and goal
ax.plot(all_2d[0, 0], all_2d[0, 1], "*", markersize=20, color="green", label="Start")
ax.plot(all_2d[-1, 0], all_2d[-1, 1], "*", markersize=20, color="red", label="Goal")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Planned trajectory in latent space")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Planning with Discrete Actions

WorldKit handles both continuous and discrete action spaces.
For environments like CartPole (push left or right), the planner
works over discrete choices.

In [ ]:
# Load a CartPole model (or use the one from notebook 02)
try:
    cartpole = WorldModel.from_hub("DilpreetBansi/cartpole-base", device="cpu")
except Exception:
    print("Hub not available, training a quick model instead")
    import gymnasium as gym
    from worldkit.data import Recorder
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    rec = Recorder(env, output="cartpole_plan.h5")
    rec.record(episodes=100, policy="random", max_steps_per_episode=200)
    cartpole = WorldModel.train(
        data="cartpole_plan.h5", config="nano",
        action_dim=1, epochs=10, device="auto"
    )

print(f"CartPole model: {cartpole.num_params:,} params")

In [ ]:
import gymnasium as gym

# Get start and goal frames from the actual environment
env = gym.make("CartPole-v1", render_mode="rgb_array")

env.reset(seed=42)
start_frame = env.render()

# Goal: push right several times to get a tilted state
env.reset(seed=42)
for _ in range(8):
    env.step(1)
goal_frame = env.render()
env.close()

# Plan with discrete action space
plan = cartpole.plan(
    current_state=start_frame,
    goal_state=goal_frame,
    max_steps=10,
    n_candidates=200,
    n_elite=20,
    n_iterations=5,
)

print(f"Planned {len(plan.actions)} actions")
print(f"Actions: {[np.array(a).flatten().tolist() for a in plan.actions]}")
print(f"Expected cost: {plan.expected_cost:.4f}")

## Key Takeaways

- **No reward function needed**: planning uses latent distance to the goal
- **Fast**: all simulation happens in latent space (no pixel rendering)
- **Flexible**: works with any action space (continuous, discrete)
- **Hierarchical**: long-horizon tasks benefit from subgoal decomposition
- **Tunable**: more candidates = better plans but slower

**When to use each planner:**

| Planner | Horizon | Speed | Use case |
|---------|---------|-------|----------|
| `model.plan()` | Short (5-30 steps) | Fast | Reactive control, simple goals |
| `model.hierarchical_plan()` | Long (30+ steps) | Slower | Navigation, multi-step tasks |

## Next Steps

| Notebook | What's next |
|----------|-------------|
| [01 — Quickstart](01_quickstart.ipynb) | Review the full WorldKit API |
| [02 — Train Custom Env](02_train_custom_env.ipynb) | Train on your own environment |
| [04 — Latent Probing](04_latent_probing.ipynb) | Understand what drives planning quality |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Deploy your planner to production |

**Advanced topics:**
- Use `model.enable_online_learning()` to improve the model during deployment
- Combine planning with `model.plausibility()` to filter unsafe action sequences
- Use `WorldModel.distill()` to create faster models for real-time planning